# Assignment 2 — Time-Series Clustering of Electricity Demand

**Course:** Machine Learning (2026W)  
**Author:** Will Sutherland  
**Dataset:** [UCI ElectricityLoadDiagrams 2011–2014](https://archive.ics.uci.edu/ml/datasets/ElectricityLoadDiagrams20112014) — 15-minute electricity demand for 370 Portuguese clients

---

## Overview

This notebook applies **unsupervised clustering** to electricity consumption time series, exploring typical usage patterns across a large client population and drilling into a single client's weekday vs. weekend behaviour.

**Key topics covered:**
- Time-series feature engineering (average normalized hourly demand curves)
- Optimal cluster selection via the **Elbow Method** and **Silhouette Score**
- **K-Means** and **Agglomerative Clustering**
- Cluster characterization and business interpretation
- Single-client analysis: linking clusters to calendar features (weekday vs. weekend)


## 0 · Imports & Data Loading

**Data note:** Download `LD2011_2014.txt` from the [UCI repository](https://archive.ics.uci.edu/ml/datasets/ElectricityLoadDiagrams20112014) and update `DATA_PATH` below. The raw file uses commas as decimal separators (European locale), which we convert on load.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from itertools import compress
from datetime import date, timedelta

from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score

import random
random.seed(42)
np.random.seed(42)

# ── Update this path to point to your local copy of the data file ───────────
DATA_PATH = r"LD2011_2014.txt"


In [ ]:
# ── Fix decimal separator and load ─────────────────────────────────────────
# The raw file uses commas as decimal points (e.g. 2,3445 → 2.3445)
import fileinput, os, shutil

fixed_path = DATA_PATH.replace(".txt", "_fixed.txt")
if not os.path.exists(fixed_path):
    with open(DATA_PATH) as src, open(fixed_path, "w") as dst:
        for line in src:
            dst.write(line.replace(",", "."))

data = pd.read_csv(fixed_path, sep=";", index_col=0, parse_dates=True)
print(f"Loaded: {data.shape[0]:,} rows × {data.shape[1]} clients")
print(f"Date range: {data.index[0]} → {data.index[-1]}")


## 1 · Data Inspection

The dataset contains one row per 15-minute interval (96 readings/day) across 4 years and 370 clients. Some clients have zero demand in early years and are excluded from the analysis.

In [ ]:
display(data.head(3))
print("Shape:", data.shape)
print("Expected rows (96 intervals × 365.25 days × 4 years ≈):", int(96 * 365.25 * 4))


In [ ]:
# ── Plot 2 days of demand for 2 clients ────────────────────────────────────
data_ex = data.loc["2012-01-01 00:15":"2012-01-03 00:00", ["MT_001", "MT_002"]]

fig, ax = plt.subplots(figsize=(12, 4))
for col in data_ex.columns:
    ax.plot(data_ex.index, data_ex[col], label=col)

ax.xaxis.set_major_locator(mdates.HourLocator(interval=4))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d\n%H:%M"))
ax.set_xlabel("Date / Time")
ax.set_ylabel("Demand (kW)")
ax.set_title("15-Minute Electricity Demand — Two Clients, Jan 1–2 2012")
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()


## 2 · Feature Engineering — Average Normalized Demand Curves

**Why normalize?** Raw demand levels vary enormously between clients (industrial vs. residential). Normalizing each client's curve by its mean allows clustering to capture *shape* (i.e., when during the day they consume) rather than *magnitude*.

**Why 2013–2014?** These years have the fewest clients with zero demand, giving the cleanest signal.


In [ ]:
# ── Filter to 2013–2014 and remove zero-demand clients ─────────────────────
data2013 = data.loc["2013-01-01 00:15":"2014-01-01 00:00"]
data2014 = data.loc["2014-01-01 00:15":"2015-01-01 00:00"]
data_13_14 = pd.concat([data2013, data2014])

zero_clients = data2013.columns[data2013.mean() == 0]
data_13_14 = data_13_14.drop(columns=zero_clients)

print(f"Clients after removing zero-demand: {data_13_14.shape[1]}")
print(f"Zero-demand clients removed: {len(zero_clients)}")


In [ ]:
# ── Compute average hourly demand per client ────────────────────────────────
data_feat = data_13_14.copy()
data_feat["hour"] = data_feat.index.hour
average_curves = data_feat.groupby("hour").mean()

# Normalize each curve to mean = 1
average_curves_norm = average_curves / average_curves.mean()

print("Normalized curves shape (24 hours × n_clients):", average_curves_norm.shape)


In [ ]:
# ── Spot-check: plot 4 normalized curves ───────────────────────────────────
average_curves_norm[["MT_001", "MT_002", "MT_369", "MT_370"]].plot(figsize=(10, 4))
plt.xlabel("Hour of Day")
plt.ylabel("Normalized Demand (mean = 1)")
plt.title("Sample Normalized Average Demand Curves")
plt.tight_layout()
plt.show()


## 3 · Clustering — All Clients

### 3.1 — Optimal k Selection (Elbow + Silhouette)

We prepare the feature matrix `X` of shape `(n_clients, 24)` — each row is a client's hourly profile.

In [ ]:
X_all = np.array(average_curves_norm.T)
print("Feature matrix shape (clients × hours):", X_all.shape)


In [ ]:
# ── Elbow method ───────────────────────────────────────────────────────────
inertia = []
K_range = range(1, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=123, n_init=10)
    km.fit(X_all)
    inertia.append(km.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(list(K_range), inertia, marker="o")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia (WCSS)")
plt.title("Elbow Method — All Clients")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# ── Silhouette scores ──────────────────────────────────────────────────────
print(f"{'k':>4}  {'Silhouette':>12}")
for k in range(2, 10):
    km = KMeans(n_clusters=k, random_state=123, n_init=10)
    labels = km.fit_predict(X_all)
    score = silhouette_score(X_all, labels)
    print(f"{k:>4}  {score:>12.4f}")


**Both methods agree on k = 4.** The elbow flattens after k = 4 and the silhouette score peaks there.

### 3.2 — Fit k = 4 and Visualize Clusters

In [ ]:
K_OPTIMAL = 4
km_all = KMeans(n_clusters=K_OPTIMAL, random_state=123, n_init=10)
labels_all = km_all.fit_predict(X_all)
centroids_all = km_all.cluster_centers_

hours = np.arange(24)

for i in range(K_OPTIMAL):
    cluster_curves = X_all[labels_all == i]
    centroid = centroids_all[i]

    plt.figure(figsize=(7, 4))
    for curve in cluster_curves:
        plt.plot(hours, curve, color="steelblue", alpha=0.25, linewidth=0.8)
    plt.plot(hours, centroid, color="firebrick", linewidth=2.5, label="Centroid")

    plt.title(f"Cluster {i+1}  ({len(cluster_curves)} clients)")
    plt.xlabel("Hour of Day")
    plt.ylabel("Normalized Demand")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


**Cluster interpretation:**

| Cluster | Size | Pattern |
|---|---|---|
| 1 | Largest | Typical household — low overnight, steady daytime rise, evening peak |
| 2 | Medium | Strong evening spike — active after-work households |
| 3 | Medium | Sharp mid-morning ramp, sustained high consumption — likely work-from-home |
| 4 | 4 clients | Anomalous early-morning peak, low daytime — specialized or unusual usage |

Cluster 4 is a natural candidate for follow-up investigation (industrial process, data quality issue, etc.).


## 4 · Single-Client Analysis — MT_022

### 4.1 — Daily Profiles and Optimal k

We now extract all 730 individual day-level demand curves for a single client and cluster them to uncover that client's distinct usage modes.

In [ ]:
CLIENT = "MT_022"
client_series = data_13_14[CLIENT]

# Build (n_days, 96) matrix — one row per day of 15-min readings
n_days = 2 * 365
X_client = np.array([
    np.array(client_series.iloc[d*96 : (d+1)*96])
    for d in range(n_days)
])
print(f"Client feature matrix shape (days × intervals): {X_client.shape}")


In [ ]:
# ── Elbow + Silhouette for single client ───────────────────────────────────
inertia_c = []
for k in range(1, 10):
    km = KMeans(n_clusters=k, random_state=123, n_init=10)
    km.fit(X_client)
    inertia_c.append(km.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(range(1, 10), inertia_c, marker="o")
plt.xlabel("k")
plt.ylabel("Inertia")
plt.title(f"Elbow Method — {CLIENT}")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n{'k':>4}  {'Silhouette':>12}")
for k in range(2, 10):
    km = KMeans(n_clusters=k, random_state=123, n_init=10)
    labels = km.fit_predict(X_client)
    score = silhouette_score(X_client, labels)
    print(f"{k:>4}  {score:>12.4f}")


### 4.2 — Fit k = 2 and Visualize

In [ ]:
K_CLIENT = 2
km_client = KMeans(n_clusters=K_CLIENT, random_state=123, n_init=10)
labels_client = km_client.fit_predict(X_client)
centroids_client = km_client.cluster_centers_

time_index = np.arange(96) / 4  # convert 15-min slots to hours

for i in range(K_CLIENT):
    cluster_curves = X_client[labels_client == i]
    centroid = centroids_client[i]

    plt.figure(figsize=(9, 4))
    for curve in cluster_curves:
        plt.plot(time_index, curve, color="steelblue", alpha=0.15, linewidth=0.6)
    plt.plot(time_index, centroid, color="firebrick", linewidth=2.5, label="Centroid")
    plt.title(f"Cluster {i+1}  ({len(cluster_curves)} days)")
    plt.xlabel("Hour of Day")
    plt.ylabel("Demand")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


### 4.3 — Linking Clusters to Calendar (Weekday vs. Weekend)

In [ ]:
# Build ordered list of date strings for the 730 days
d_start = date(2013, 1, 1)
day_labels = []
DOW = {0:"mon", 1:"tue", 2:"wed", 3:"thu", 4:"fri", 5:"sat", 6:"sun"}
for i in range(n_days):
    d = d_start + timedelta(days=i)
    day_labels.append(f"{DOW[d.weekday()]}-{d}")

def day_type(day_str):
    return "Weekend" if ("sat" in day_str or "sun" in day_str) else "Weekday"

cluster0_days = list(compress(day_labels, labels_client == 0))
cluster1_days = list(compress(day_labels, labels_client == 1))

summary = pd.DataFrame({
    f"Cluster 1 ({len(cluster0_days)} days)": pd.Series([day_type(d) for d in cluster0_days]).value_counts(),
    f"Cluster 2 ({len(cluster1_days)} days)": pd.Series([day_type(d) for d in cluster1_days]).value_counts(),
})
print(summary)
print("\nProportions:")
print((summary / summary.sum()).round(3))


## 5 · Summary & Key Takeaways

**All-client clustering (k = 4)** revealed four distinct electricity usage archetypes ranging from typical residential patterns to specialized early-morning usage. Cluster 4 (4 clients) is an anomaly worth flagging to operations.

**Single-client clustering (k = 2)** cleanly separated weekday and weekend behaviour. The larger cluster (Cluster 2, ~511 days) corresponds predominantly to weekdays, with a structured morning ramp and stable afternoon plateau. The smaller cluster (Cluster 1, ~219 days) captures weekend days with more consistent all-day consumption.

**Takeaway:** Normalizing curves before clustering is essential — without it, K-Means would cluster by *consumption level* rather than *consumption pattern*, which is the more actionable signal for demand management and tariff design.
